# Notebook 04 — GeoAI in Action: From Pipeline to Discovery
**GeoAI for Climate, Environment and Health** | 25 June 2026

---

> **This is the payoff notebook.** Notebooks 01–03 built the full pipeline piece by piece:
> you loaded real Zimbabwe DHS 2015 birth records, explored satellite climate data,
> constructed trimester-level exposure windows. Every step required careful Python.
>
> ```
> 📋 Health records → 🛰 Satellite → 🌍 Environment → 🔗 Link → ⏱ Timing → [🎯 GeoAI in Action]
> ```
>
> **The question:** *Do those environmental conditions predict birth outcomes?*
>
> We answer it two ways — **manually** (forest plot + maps, so you understand the statistics)
> and then with **GeoAI** (so you see what becomes possible when the coding barrier disappears).

---

## Learning objectives
- Map adverse birth outcomes across Zimbabwe provinces and identify spatial patterns
- Fit and interpret a logistic regression: P(LBW) ~ heat + drought + PM2.5 + socioeconomic factors
- Read odds ratios and 95% confidence intervals in a forest plot
- Use GeoAI to replicate and extend the same analyses in plain English


In [ ]:
import pathlib
if not (pathlib.Path.home() / '.nb03_complete').exists():
    raise SystemExit("Complete Notebook 03 (Exposure Timeline Characterisation) first.")
print("\u2705  Prerequisite check passed \u2014 welcome to Notebook 04.")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
import statsmodels.api as sm

import pathlib, sys

def _find_data():
    here = pathlib.Path().resolve()
    for p in (here, here.parent, here.parent.parent):
        if (p / 'data').is_dir():
            if str(p) not in sys.path:
                sys.path.insert(0, str(p))
            return p / 'data'
    raise FileNotFoundError('Cannot find data/ directory')

DATA_DIR = _find_data()
print(f'Data directory: {DATA_DIR}')

def mc_question(question, options, correct, explanation):
    try:
        import ipywidgets as w
        from IPython.display import display, HTML
        header = w.HTML(f'<div style="font-size:15px;font-weight:600;margin-bottom:10px">🧠 {question}</div>')
        radio = w.RadioButtons(options=[f"{k})  {v}" for k, v in options.items()],
                               layout=w.Layout(width="100%"),
                               style={"description_width": "0px"})
        feedback = w.Output()
        def _check(change):
            if change["name"] != "value": return
            letter = change["new"][0]
            with feedback:
                feedback.clear_output(wait=True)
                if letter == correct:
                    display(HTML(f'<div style="background:#d4edda;color:#155724;padding:10px 14px;border-radius:6px;margin-top:6px;border-left:4px solid #28a745">✅ <b>Correct!</b>&nbsp; {explanation}</div>'))
                else:
                    display(HTML(f'<div style="background:#f8d7da;color:#721c24;padding:10px 14px;border-radius:6px;margin-top:6px;border-left:4px solid #dc3545">❌ <b>Not quite.</b>&nbsp; The answer is <b>{correct}</b>. {explanation}</div>'))
        radio.observe(_check, names="value")
        display(w.VBox([header, radio, feedback],
                       layout=w.Layout(border="1px solid #dee2e6", padding="16px 20px",
                                       border_radius="8px", margin="16px 0")))
    except ImportError:
        print(f"Q: {question}")
        for k, v in options.items():
            print(f"  {k}) {v}{' ← CORRECT' if k == correct else ''}")
        print(f"Explanation: {explanation}")

import duckdb

# Load core data
# Each student's trimester file is in their home directory (written by Notebook 05).
# If it exists there we use it; otherwise we fall back to the shared pre-generated copy.
import pathlib as _pl
_home_trimester = _pl.Path.home() / 'zw_births_trimester.csv'
_shared_trimester = DATA_DIR / 'zw_births_trimester.csv'
_trimester_path = _home_trimester if _home_trimester.exists() else _shared_trimester
b         = pd.read_csv(_trimester_path)
print(f'Loaded trimester data from: {_trimester_path}')
clusters  = pd.read_csv(DATA_DIR / 'zw_clusters.csv')
climate   = pd.read_csv(DATA_DIR / 'zw_climate_monthly.csv')

# Annual mean climate per cluster — climate means only; cluster metadata stays in `clusters`
clim_ann = (
    climate
    .groupby('cluster_id')[['temp_c', 'rain_mm', 'ndvi', 'pm25', 'dust_aod550']]
    .mean()
    .reset_index()
    .rename(columns={'temp_c': 'mean_temp', 'rain_mm': 'mean_rain', 'ndvi': 'mean_ndvi',
                     'pm25': 'mean_pm25', 'dust_aod550': 'mean_dust'})
)

# Birth records with valid birth weight for regression
b_lbw = b[b['low_birth_weight'].notna()].copy()

print(f"Births loaded: {len(b):,}  |  with LBW flag: {len(b_lbw):,}")
print(f"Clusters: {len(clusters)}  |  Climate rows: {len(climate):,}")
print(f"clim_ann cols: {list(clim_ann.columns)}")
# Cluster-level merged frame — available to all cells
cl_merged = clusters.merge(clim_ann, on='cluster_id')
print(f'cl_merged: {len(cl_merged)} clusters')


In [ ]:
prov = b.groupby('province').agg(
    n_births         = ('neonatal_death', 'count'),
    lbw_n            = ('low_birth_weight', 'sum'),
    lbw_denom        = ('low_birth_weight', lambda x: x.notna().sum()),
    neonatal_deaths  = ('neonatal_death', 'sum'),
    mean_temp_t3     = ('temp_t3', 'mean'),
    mean_ndvi_t3     = ('ndvi_t3', 'mean'),
    pct_urban        = ('urban', 'mean'),
).reset_index()

prov['lbw_pct']         = (prov['lbw_n'] / prov['lbw_denom'] * 100).round(1)
prov['neonatal_per1000'] = (prov['neonatal_deaths'] / prov['n_births'] * 1000).round(1)

display_cols = prov[['province','lbw_pct','neonatal_per1000','mean_temp_t3','mean_ndvi_t3','pct_urban']].copy()
display_cols['pct_urban']     = (display_cols['pct_urban']*100).round(0)
display_cols['mean_temp_t3']  = display_cols['mean_temp_t3'].round(1)
display_cols['mean_ndvi_t3']  = display_cols['mean_ndvi_t3'].round(3)
display_cols.columns = ['Province','LBW %','Neonatal MR/1000','Mean T3 Temp (°C)','Mean T3 NDVI','% Urban']
print(display_cols.sort_values('LBW %', ascending=False).to_string(index=False))

---
## Part A — Where are adverse outcomes concentrated?

Before modelling, we map the spatial distribution of birth outcomes.
Zimbabwe spans dramatic gradients: cool Eastern Highlands (Manicaland),
hot dry lowveld (Masvingo, Matabeleland South), and urban centres (Harare, Bulawayo).

The maps below establish the spatial baseline. Part B (GeoAI) will reproduce
the same analysis in one sentence — that contrast is the point.


In [ ]:
# Cluster dot maps: LBW rate (left) and mean PM2.5 air pollution (right)
cl_lbw = cl_merged[cl_merged['lbw_denom'] >= 5].copy()

import contextily as ctx
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sc1 = axes[0].scatter(cl_lbw['cluster_lon'], cl_lbw['cluster_lat'],
    c=cl_lbw['lbw_rate']*100, cmap='YlOrRd', s=30, alpha=0.8, vmin=0, vmax=25)
plt.colorbar(sc1, ax=axes[0], label='LBW rate (%)')
axes[0].set_title('Cluster LBW Rate', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')

sc2 = axes[1].scatter(cl_merged['cluster_lon'], cl_merged['cluster_lat'],
    c=cl_merged['mean_pm25'], cmap='YlOrBr', s=30, alpha=0.8)
plt.colorbar(sc2, ax=axes[1], label='Mean PM2.5 (\u00b5g/m\u00b3)')
axes[1].set_title('Mean PM2.5 Air Pollution 2011\u20132015', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Longitude'); axes[1].set_ylabel('Latitude')

plt.suptitle('Zimbabwe DHS 2015 \u2014 Cluster LBW Rate vs PM2.5 Air Pollution',
             fontsize=12, fontweight='bold')
ctx.add_basemap(axes[0], crs='EPSG:4326', source=ctx.providers.CartoDB.Positron, zoom=6, alpha=0.6)
ctx.add_basemap(axes[1], crs='EPSG:4326', source=ctx.providers.CartoDB.Positron, zoom=6, alpha=0.6)
plt.tight_layout(); plt.show()
print('Do high-LBW clusters (dark red) appear where PM2.5 is highest (dark brown)?')


### Cluster-level ecological correlations

The scatter plots below show how NDVI (vegetation greenness) and temperature
correlate with LBW rate at the cluster level.
*(Code is hidden — run the cell to see the charts.)*


In [ ]:
cl_eco = cl_merged[cl_merged['lbw_denom'] >= 5].copy()

r_ndvi_lbw_raw, p_ndvi = stats.pearsonr(cl_eco['mean_ndvi'], cl_eco['lbw_rate'])
r_temp_lbw_raw, p_temp  = stats.pearsonr(cl_eco['mean_temp'],  cl_eco['lbw_rate'])
r_ndvi_lbw = round(r_ndvi_lbw_raw, 3)
r_temp_lbw  = round(r_temp_lbw_raw,  3)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, xcol, r_val, p_val, xlabel in [
    (axes[0], 'mean_ndvi',  r_ndvi_lbw, p_ndvi, 'Mean annual NDVI'),
    (axes[1], 'mean_temp',  r_temp_lbw, p_temp, 'Mean annual temperature (°C)'),
]:
    x = cl_eco[xcol]; y = cl_eco['lbw_rate'] * 100
    ax.scatter(x, y, alpha=0.4, s=15, color='#2a9d8f')
    m, b_ = np.polyfit(x, y, 1)
    xline = np.linspace(x.min(), x.max(), 100)
    ax.plot(xline, m*xline+b_, 'r-', lw=2)
    ax.set_xlabel(xlabel); ax.set_ylabel('Cluster LBW rate (%)')
    ax.set_title(f'r = {r_val:.3f}  (p = {p_val:.3f})', fontsize=11)

plt.suptitle('Cluster-level Environmental Correlates of Low Birth Weight', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

print(f'r(NDVI, LBW rate)  = {r_ndvi_lbw}  (p={p_ndvi:.3f})')
print(f'r(Temp, LBW rate)  = {r_temp_lbw}   (p={p_temp:.3f})')

In [ ]:
%%html
<div style="font-family:system-ui,sans-serif;border:1px solid #dee2e6;border-radius:8px;padding:16px 20px;margin:16px 0;background:#fafafa">
  <div style="font-size:15px;font-weight:600;margin-bottom:12px">🧠 The regression shows higher T3 LST is associated with increased LBW odds. What alternative explanation (confounding) should you always consider?</div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb04q1" value="A" style="margin-top:3px;flex-shrink:0"><span><b>A)</b> The regression model may have a coding error</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb04q1" value="B" style="margin-top:3px;flex-shrink:0"><span><b>B)</b> Hot, arid areas also tend to have higher poverty and food insecurity, which independently increase LBW risk — heat and poverty are co-located</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb04q1" value="C" style="margin-top:3px;flex-shrink:0"><span><b>C)</b> LST directly causes LBW with no mediating pathway</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb04q1" value="D" style="margin-top:3px;flex-shrink:0"><span><b>D)</b> The association is only present in urban clusters</span></label></div>
  <div id="nb04q1-fb" style="margin-top:10px"></div>
</div>
<script>
(function(){
  var C="B",E="This is the classic confounding problem in ecological studies. Temperature and socioeconomic deprivation are spatially correlated in Zimbabwe. Controlling for wealth index partially addresses this, but residual confounding is always possible.",ID="nb04q1";
  function showFeedback(val){
    var fb=document.getElementById(ID+"-fb");
    if(!fb)return;
    if(val===C){
      fb.innerHTML='<div style="background:#d4edda;color:#155724;padding:10px 14px;border-radius:6px;border-left:4px solid #28a745">✅ <b>Correct!</b> '+E+'</div>';
    }else{
      fb.innerHTML='<div style="background:#f8d7da;color:#721c24;padding:10px 14px;border-radius:6px;border-left:4px solid #dc3545">❌ <b>Not quite.</b> The answer is <b>'+C+'</b>. '+E+'</div>';
    }
    document.dispatchEvent(new CustomEvent('mcq-update'));
  }
  function attach(){
    var saved=localStorage.getItem('mcq_'+ID);
    document.querySelectorAll('input[name="'+ID+'"]').forEach(function(r){
      if(saved&&r.value===saved){r.checked=true;}
      r.addEventListener("change",function(){
        localStorage.setItem('mcq_'+ID,this.value);
        showFeedback(this.value);
      });
    });
    if(saved)showFeedback(saved);
  }
  if(document.readyState==="loading"){document.addEventListener("DOMContentLoaded",attach);}else{attach();}
})();
</script>

---
## Logistic regression and the forest plot

Low birth weight is a binary outcome. We fit a logistic regression predicting
P(LBW) from trimester exposures and socioeconomic factors.

> An **odds ratio (OR)** of 1.0 = no association. OR > 1 = higher odds of LBW.
> OR < 1 = lower odds (protective). The 95% CI tells us the range of plausible values.
> If the CI crosses 1.0, the association is **not statistically significant** at α = 0.05.

The forest plot below displays all predictors at a glance — the standard display
in epidemiology.


In [ ]:
mod_df = b_lbw.copy()
mod_df['edu_num'] = mod_df['education'].map({'none':0,'primary':1,'secondary':2,'higher':3})

# Standardise continuous predictors for comparable coefficients
for col in ['temp_t3', 'rain_t3', 'ndvi_t3', 'pm25_t3', 'dust_t3', 'mother_age']:
    mu, sd = mod_df[col].mean(), mod_df[col].std()
    mod_df[f'{col}_z'] = (mod_df[col] - mu) / sd

X_cols = ['temp_t3_z', 'rain_t3_z', 'ndvi_t3_z', 'pm25_t3_z', 'dust_t3_z',
          'urban', 'wealth_quintile', 'birth_order', 'edu_num', 'mother_age_z']
X = sm.add_constant(mod_df[X_cols].astype(float))
y = mod_df['low_birth_weight'].astype(float)

# Drop rows with any missing predictor or outcome
valid = X.notna().all(axis=1) & y.notna()
X, y = X[valid], y[valid]
print(f'Fitting on {len(y):,} births (dropped {(~valid).sum()} with missing values)')

model = sm.Logit(y, X).fit(disp=False)

odds   = np.exp(model.params[1:])
ci_lo  = np.exp(model.conf_int().iloc[1:, 0])
ci_hi  = np.exp(model.conf_int().iloc[1:, 1])
pvals  = model.pvalues[1:]

labels = [
    'T3 temperature (+1 SD)', 'T3 rainfall (+1 SD)', 'T3 NDVI (+1 SD)',
    'T3 PM2.5 (+1 SD)', 'T3 dust AOD (+1 SD)',
    'Urban cluster', 'Wealth quintile (+1)',
    'Birth order (+1)', 'Education (+1 level)', 'Maternal age (+1 SD)',
]

results = pd.DataFrame({'OR': odds.values, 'CI_lo': ci_lo.values, 'CI_hi': ci_hi.values,
                         'p': pvals.values}, index=labels)
results['Sig'] = results['p'].apply(
    lambda p: '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else '')))
results['OR']    = results['OR'].round(3)
results['CI_lo'] = results['CI_lo'].round(3)
results['CI_hi'] = results['CI_hi'].round(3)
results['p']     = results['p'].round(3)

print(results.to_string())
print(f"\nPseudo R-squared (McFadden): {model.prsquared:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
y_pos = np.arange(len(results))

colours = ['#2a9d8f' if ch < 1 else ('#e63946' if cl > 1 else '#888888')
           for cl, ch in zip(results['CI_lo'], results['CI_hi'])]

# Draw error bars without markers, then scatter markers individually for per-point colour
ax.errorbar(
    results['OR'], y_pos,
    xerr=[results['OR']-results['CI_lo'], results['CI_hi']-results['OR']],
    fmt='none', ecolor='k', capsize=4, capthick=1.5, lw=1.2,
)
for xi, yi, c in zip(results['OR'], y_pos, colours):
    ax.plot(xi, yi, 'o', markersize=8, markerfacecolor=c, markeredgecolor='k', markeredgewidth=0.8)

ax.axvline(1.0, color='black', linestyle='--', lw=1.2)
ax.set_yticks(y_pos)
ax.set_yticklabels(results.index, fontsize=10)
ax.set_xlabel('Odds Ratio (95% CI)', fontsize=11)
ax.set_title('Predictors of Low Birth Weight — Zimbabwe DHS 2015\nLogistic regression, n = {:,}'.format(len(y)),
             fontsize=11, fontweight='bold')
ax.set_xlim(0.45, 1.85)

handles = [
    mpatches.Patch(color='#2a9d8f', label='OR < 1 (protective — CI excludes 1.0)'),
    mpatches.Patch(color='#e63946', label='OR > 1 (risk — CI excludes 1.0)'),
    mpatches.Patch(color='#888888', label='CI crosses 1.0 (inconclusive)'),
]
ax.legend(handles=handles, loc='lower right', fontsize=9)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

> **Findings.** Significant predictors: maternal education (protective), birth order (protective),
> maternal age (slight risk). Environmental predictors (heat, rainfall, NDVI) are in the expected
> directions but confidence intervals cross 1.0.

> **Why?** Three reasons: (1) cluster jitter means exposure is imprecise;
> (2) birth weight is only recorded for ~30% of births — selection is non-random
> (facility-delivered, urban births overrepresented); (3) monthly CMC birth date gives
> only a rough proxy for trimester exposure timing, introducing noise.
> **Environmental epidemiology: study design and data quality determine what we detect.**


In [ ]:
%%html
<div style="font-family:system-ui,sans-serif;border:1px solid #dee2e6;border-radius:8px;padding:16px 20px;margin:16px 0;background:#fafafa">
  <div style="font-size:15px;font-weight:600;margin-bottom:12px">🧠 The GeoAI assistant shows you the SQL it generated before running it. Why is this transparency important?</div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb04q2" value="A" style="margin-top:3px;flex-shrink:0"><span><b>A)</b> Databases require human approval before executing any query</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb04q2" value="B" style="margin-top:3px;flex-shrink:0"><span><b>B)</b> Showing the SQL lets you verify that the query actually answers the question you asked — AI can misinterpret ambiguous natural language</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb04q2" value="C" style="margin-top:3px;flex-shrink:0"><span><b>C)</b> SQL is decorative and helps the interface look more technical</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb04q2" value="D" style="margin-top:3px;flex-shrink:0"><span><b>D)</b> Transparency is required by GDPR</span></label></div>
  <div id="nb04q2-fb" style="margin-top:10px"></div>
</div>
<script>
(function(){
  var C="B",E="AI-generated SQL should always be treated as a draft. Reviewing it closes the loop between your intent and the computer\'s interpretation — a critical skill as AI tools become routine in research workflows.",ID="nb04q2";
  function showFeedback(val){
    var fb=document.getElementById(ID+"-fb");
    if(!fb)return;
    if(val===C){
      fb.innerHTML='<div style="background:#d4edda;color:#155724;padding:10px 14px;border-radius:6px;border-left:4px solid #28a745">✅ <b>Correct!</b> '+E+'</div>';
    }else{
      fb.innerHTML='<div style="background:#f8d7da;color:#721c24;padding:10px 14px;border-radius:6px;border-left:4px solid #dc3545">❌ <b>Not quite.</b> The answer is <b>'+C+'</b>. '+E+'</div>';
    }
    document.dispatchEvent(new CustomEvent('mcq-update'));
  }
  function attach(){
    var saved=localStorage.getItem('mcq_'+ID);
    document.querySelectorAll('input[name="'+ID+'"]').forEach(function(r){
      if(saved&&r.value===saved){r.checked=true;}
      r.addEventListener("change",function(){
        localStorage.setItem('mcq_'+ID,this.value);
        showFeedback(this.value);
      });
    });
    if(saved)showFeedback(saved);
  }
  if(document.readyState==="loading"){document.addEventListener("DOMContentLoaded",attach);}else{attach();}
})();
</script>

In [ ]:
%%html
<div style="font-family:system-ui,sans-serif;border:1px solid #dee2e6;border-radius:8px;padding:16px 20px;margin:16px 0;background:#fafafa">
  <div style="font-size:15px;font-weight:600;margin-bottom:12px">🧠 You have now run the complete GeoAI pipeline. What is the most important caveat about the conclusions you can draw from this analysis?</div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb04q3" value="A" style="margin-top:3px;flex-shrink:0"><span><b>A)</b> The Zimbabwe DHS dataset is too small for any conclusions</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb04q3" value="B" style="margin-top:3px;flex-shrink:0"><span><b>B)</b> These are observational ecological associations — we cannot claim that climate directly causes LBW without ruling out confounders and without individual-level exposure data</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb04q3" value="C" style="margin-top:3px;flex-shrink:0"><span><b>C)</b> Logistic regression is not valid for birth outcome data</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb04q3" value="D" style="margin-top:3px;flex-shrink:0"><span><b>D)</b> DHS data is not peer-reviewed and cannot be cited</span></label></div>
  <div id="nb04q3-fb" style="margin-top:10px"></div>
</div>
<script>
(function(){
  var C="B",E="Ecological studies identify patterns and generate hypotheses. Moving from association to causation requires individual-level data, longitudinal study designs, and rigorous confounder control.",ID="nb04q3";
  function showFeedback(val){
    var fb=document.getElementById(ID+"-fb");
    if(!fb)return;
    if(val===C){
      fb.innerHTML='<div style="background:#d4edda;color:#155724;padding:10px 14px;border-radius:6px;border-left:4px solid #28a745">✅ <b>Correct!</b> '+E+'</div>';
    }else{
      fb.innerHTML='<div style="background:#f8d7da;color:#721c24;padding:10px 14px;border-radius:6px;border-left:4px solid #dc3545">❌ <b>Not quite.</b> The answer is <b>'+C+'</b>. '+E+'</div>';
    }
    document.dispatchEvent(new CustomEvent('mcq-update'));
  }
  function attach(){
    var saved=localStorage.getItem('mcq_'+ID);
    document.querySelectorAll('input[name="'+ID+'"]').forEach(function(r){
      if(saved&&r.value===saved){r.checked=true;}
      r.addEventListener("change",function(){
        localStorage.setItem('mcq_'+ID,this.value);
        showFeedback(this.value);
      });
    });
    if(saved)showFeedback(saved);
  }
  if(document.readyState==="loading"){document.addEventListener("DOMContentLoaded",attach);}else{attach();}
})();
</script>

In [ ]:
%%html
<div style="font-family:system-ui,sans-serif;border:1px solid #dee2e6;border-radius:8px;padding:16px 20px;margin:16px 0;background:#fafafa">
  <div style="font-size:15px;font-weight:600;margin-bottom:12px">🧠 Thinking about the full pipeline you have completed — where is the role of remote sensing professionals most critical?</div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb04q4" value="A" style="margin-top:3px;flex-shrink:0"><span><b>A)</b> Only in Notebook 05 when we discuss Earth Engine</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb04q4" value="B" style="margin-top:3px;flex-shrink:0"><span><b>B)</b> Nowhere — remote sensing is background infrastructure</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb04q4" value="C" style="margin-top:3px;flex-shrink:0"><span><b>C)</b> Throughout: selecting appropriate EO products, understanding spatial resolution and temporal coverage, validating satellite-derived estimates, and communicating what the data can and cannot show</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb04q4" value="D" style="margin-top:3px;flex-shrink:0"><span><b>D)</b> Only when producing the final maps in Notebook 06</span></label></div>
  <div id="nb04q4-fb" style="margin-top:10px"></div>
</div>
<script>
(function(){
  var C="C",E="Remote sensing expertise is embedded in every stage: knowing which product to use (NB02), understanding resolution limitations (NB05), interpreting spatial patterns (NB03, NB06). This course bridges EO and health — both communities need each other.",ID="nb04q4";
  function showFeedback(val){
    var fb=document.getElementById(ID+"-fb");
    if(!fb)return;
    if(val===C){
      fb.innerHTML='<div style="background:#d4edda;color:#155724;padding:10px 14px;border-radius:6px;border-left:4px solid #28a745">✅ <b>Correct!</b> '+E+'</div>';
    }else{
      fb.innerHTML='<div style="background:#f8d7da;color:#721c24;padding:10px 14px;border-radius:6px;border-left:4px solid #dc3545">❌ <b>Not quite.</b> The answer is <b>'+C+'</b>. '+E+'</div>';
    }
    document.dispatchEvent(new CustomEvent('mcq-update'));
  }
  function attach(){
    var saved=localStorage.getItem('mcq_'+ID);
    document.querySelectorAll('input[name="'+ID+'"]').forEach(function(r){
      if(saved&&r.value===saved){r.checked=true;}
      r.addEventListener("change",function(){
        localStorage.setItem('mcq_'+ID,this.value);
        showFeedback(this.value);
      });
    });
    if(saved)showFeedback(saved);
  }
  if(document.readyState==="loading"){document.addEventListener("DOMContentLoaded",attach);}else{attach();}
})();
</script>

---

> ## The pivot moment
>
> You have now seen the manual pipeline: cluster maps, ecological correlations,
> a logistic regression, and a forest plot. Each step required knowing the right
> library, the right syntax, and the right statistical method.
>
> **Now we ask the same questions in plain English.**
>
> Part B shows what GeoAI does with this dataset — and why that matters for
> health research in settings where epidemiological coding skills are scarce.


---
## Part B — GeoAI: The same analysis in plain English

**What Part A required:** a cluster scatter map (25 lines), ecological correlations
with regression lines (20 lines), a logistic model with standardised predictors (30 lines),
a forest plot (20 lines). Approximately **95 lines of domain-specific Python**.

**What GeoAI requires:** a sentence.

That is not magic — GeoAI translates your question into the same Python (or SQL),
runs it on the same data, and returns the result. The barrier has moved: instead of
needing to know pandas, statsmodels, and matplotlib, you need to know *what question
to ask* — the scientific part, not the coding part.

**Transparency:** Every answer shows the generated code in a collapsible panel
(▶ View generated code). Always check it — GeoAI can make mistakes, and you now
have the statistical knowledge to catch them.

**Privacy:** Individual identifiers are stripped server-side before any analysis runs.


In [ ]:
import urllib.request as _ureq, json as _json, os as _os

GEOAI_URL = _os.environ.get('GEOAI_API_URL', 'http://localhost:5000/api/workshop/geoai')

def geoai(question: str) -> None:
    """Ask a plain-English question — returns a SQL query and its results.
    If the question is ambiguous the assistant will ask for clarification.
    """
    payload = _json.dumps({'question': question, 'db': 'exposures_zw'}).encode()
    try:
        req = _ureq.Request(GEOAI_URL, data=payload,
                            headers={'Content-Type': 'application/json'}, method='POST')
        with _ureq.urlopen(req, timeout=30) as resp:
            res = _json.loads(resp.read())

        # Model asked for clarification
        if res.get('clarify'):
            print(f'🤔 {res["clarify"]}')
            return

        if res.get('error') and not res.get('sql'):
            print(f'⚠️  {res["error"]}')
            return

        print('SQL generated:')
        print(res.get('sql', '(none)'))
        df = pd.DataFrame(res.get('rows', []), columns=res.get('columns', []))
        print('\nResults:')
        print(df.to_string(index=False) if len(df) > 0 else '(no rows)')
        if res.get('error'):
            print(f'\n⚠️  Note: {res["error"]}')
    except Exception as e:
        print(f'GeoAI unavailable ({e})')

geoai('Which 5 clusters have the highest LBW rate among rural clusters with at least 10 weighed births?')


In [ ]:
import urllib.request as _ureq, json as _json, os as _os
import base64
from IPython.display import display, Image, HTML

GEOAI_ANALYZE_URL = _os.environ.get('GEOAI_ANALYZE_URL', 'http://localhost:5000/api/workshop/geoai_analyze')

# Conversation history — persists across ask() calls in this notebook session
_geoai_history = []

def ask(question: str, new: bool = False) -> None:
    """
    Ask a climate-health question in plain English.
    Results appear immediately (chart first, then summary text).
    Generated code is collapsed at the bottom for reference.

    Pass new=True to reset the conversation and start fresh.

    Examples:
        ask("Plot LBW rate by province as a bar chart")
        ask("Now break that down by urban vs rural")          # follow-up
        ask("Which province has the biggest urban–rural gap?")
        ask("", new=True)                                     # fresh start
    """
    global _geoai_history

    if new:
        _geoai_history = []
        print("── New conversation started ──")
        if not question.strip():
            return

    if not question.strip():
        print('Tip: ask("your question here")  |  ask("", new=True) to reset')
        return

    payload = _json.dumps({
        'question': question,
        'history':  _geoai_history,
    }).encode()

    try:
        req = _ureq.Request(GEOAI_ANALYZE_URL, data=payload,
                            headers={'Content-Type': 'application/json'}, method='POST')
        with _ureq.urlopen(req, timeout=60) as resp:
            res = _json.loads(resp.read())
    except Exception as e:
        print(f'GeoAI unavailable: {e}')
        return

    if res.get('error') and not res.get('figures') and not res.get('output'):
        print(f'Error: {res["error"]}')
        return

    # Update conversation history so the next ask() has context
    _geoai_history = res.get('history', _geoai_history)

    # ── Results first: figures → printed output ──────────────────────────────
    for fig_b64 in res.get('figures', []):
        display(Image(data=base64.b64decode(fig_b64)))
    if res.get('output'):
        print(res['output'].rstrip())
    if res.get('error'):
        print(f'Note: {res["error"]}')

    # ── Generated code — collapsed, for those who want to dig deeper ─────────
    code = (res.get('code') or '').strip()
    if code:
        escaped = code.replace('&','&amp;').replace('<','&lt;').replace('>','&gt;')
        display(HTML(
            '<details style="margin-top:10px;font-size:12px;color:#6c757d">'
            '<summary style="cursor:pointer;user-select:none">'
            '&#9656; View generated code</summary>'
            '<pre style="background:#f6f8fa;padding:12px 16px;border-radius:6px;'
            'overflow-x:auto;margin-top:6px;font-size:11px;line-height:1.5">'
            + escaped + '</pre></details>'
        ))


### The GeoAI investigation

**Research question:** *Do environmental conditions during pregnancy affect birth
outcomes in Zimbabwe, after accounting for wealth and urbanisation?*

Run each cell in order. Before each result appears, read the comment explaining
*why* that question is the right next step.


#### Step 1 — Spatial distribution: where is LBW concentrated?

If environment matters, hotter and drier provinces (Masvingo, Matabeleland South)
should show higher LBW rates than cooler, wetter ones (Manicaland).
Compare this chart with the manual cluster maps above.


In [ ]:
# Step 1 — replaces ~20 lines of Part A code
ask("Plot LBW rate by province as a horizontal bar chart sorted highest to lowest, with a vertical line for the national average")


#### Step 2 — Urban/rural confounding: is the spatial pattern real?

Province differences in LBW could simply reflect urban composition.
We need to separate *where* from *how urban*. Note: Bulawayo has no rural births
in this dataset — you will see only one bar for it.


In [ ]:
# Step 2 — ask() automatically sends the Step 1 context
ask("Now break each province into urban vs rural as a grouped horizontal bar chart")


#### Step 3 — Which province has the widest urban–rural gap?

A large within-province gap suggests place-specific environmental factors,
not just demographics. This is where satellite-derived exposure data adds
real explanatory value.


In [ ]:
ask("Which province has the biggest urban–rural gap in LBW rate? Report the gap in percentage points and note the mean T3 temperature for that province.")


#### Step 4 — Do environmental factors survive statistical adjustment?

Patterns in bar charts can be confounded. Now ask GeoAI to produce the same
forest plot as Part A — in a single sentence. Compare the odds ratios.


In [ ]:
ask("", new=True)
ask("Run a logistic regression of low_birth_weight on temp_t3, drought_exposed, wealth_quintile, and urban. Standardise the continuous predictors. Display odds ratios as a forest plot with 95% CIs.")


#### Your turn — extend the investigation

The four steps above answered the core research question. Now push further:
- *"Show LBW rate by birth month — is there a seasonal pattern?"*
- *"Compare mean T3 NDVI for LBW vs normal-weight births by province"*
- *"Which wealth quintile has the highest neonatal mortality rate?"*
- *"Scatter plot: cluster T3 temperature vs cluster LBW rate, coloured by urban"*


In [ ]:
# Replace this with your own question
ask("Show the seasonality of LBW rate by birth month, with mean T3 temperature on a twin y-axis")


---
## GeoAI Chat: Explore freely

The investigation above used `ask()` in a structured script. The chat widget below
is open-ended: type any question about the Zimbabwe birth outcomes dataset.
Each answer is generated fresh; the **▶ View generated code** panel shows exactly what
Python the AI wrote — check it against what you now know from Part A.


---
## GeoAI Chat — Interactive Research Assistant

Use the chat below to explore the Zimbabwe DHS 2015 data in plain English. Ask questions, request plots, or run statistical analyses — the assistant writes and executes the code for you.

> **▶ Run the cell below to launch the chat.** If you see a "model not found" message, simply re-run the cell (Shift + Enter) to get a fresh widget.


In [ ]:
import ipywidgets as _w
from IPython.display import display as _disp, Image as _Img, HTML as _HTML
import json as _json, base64 as _b64, urllib.request as _ureq, os as _os

def make_geoai_chat():
    _url  = _os.environ.get('GEOAI_ANALYZE_URL', 'http://localhost:5000/api/workshop/geoai_analyze')
    _hist = []

    header = _w.HTML("""
<div style="background:linear-gradient(135deg,#1b7a4b 0%,#1565c0 100%);
            color:white;padding:13px 18px;border-radius:8px 8px 0 0;
            font-family:system-ui,sans-serif;display:flex;align-items:center;gap:10px">
  <span style="font-size:22px">🤖</span>
  <div>
    <div style="font-weight:700;font-size:14px;letter-spacing:.01em">Shmron</div>
    <div style="font-size:11px;opacity:.85">Zimbabwe DHS 2015 · births · climate · exposures</div>
  </div>
</div>""")

    chat = _w.Output(layout=_w.Layout(
        min_height='360px', max_height='520px',
        overflow_y='auto', padding='14px 16px',
        background_color='#ffffff',
    ))
    status = _w.Output(layout=_w.Layout(min_height='24px'))

    q_input = _w.Text(
        placeholder='Ask about births, exposures, provinces, regressions…',
        layout=_w.Layout(flex='1', min_width='0', height='34px'),
    )
    send_btn = _w.Button(
        description='Send ↑', button_style='success',
        layout=_w.Layout(width='82px', height='34px'),
    )
    new_btn = _w.Button(
        description='+ New', tooltip='Start a fresh conversation',
        layout=_w.Layout(width='68px', height='34px'),
    )
    footer = _w.HBox(
        [q_input, send_btn, new_btn],
        layout=_w.Layout(padding='8px 12px', gap='6px',
                         border_top='1px solid #dee2e6',
                         background_color='#f8f9fa'),
    )

    def _user_bubble(text):
        return (
            '<div style="text-align:right;margin:10px 0;font-family:system-ui,sans-serif">'
            f'<span style="background:#0d6efd;color:#fff;padding:8px 14px;'
            'border-radius:18px 18px 4px 18px;display:inline-block;'
            f'max-width:82%;text-align:left;font-size:13px">{text}</span></div>'
        )

    def _bot_bubble(text):
        return (
            '<div style="margin:10px 0;font-family:system-ui,sans-serif">'
            '<div style="display:flex;align-items:flex-start;gap:8px">'
            '<span style="font-size:18px;flex-shrink:0;margin-top:2px">🤖</span>'
            '<div style="background:#f1f3f5;padding:10px 14px;'
            'border-radius:4px 18px 18px 18px;max-width:85%;'
            f'font-size:13px;line-height:1.55">{text}</div>'
            '</div></div>'
        )

    def _code_block(code):
        esc = code.replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;')
        return (
            '<details style="margin:4px 0 6px 26px;font-size:11px;'
            'color:#6c757d;font-family:system-ui,sans-serif">'
            '<summary style="cursor:pointer;user-select:none">▶ View generated code</summary>'
            '<pre style="background:#f6f8fa;padding:10px 12px;border-radius:6px;'
            f'overflow-x:auto;font-size:10.5px;line-height:1.45;margin-top:4px">{esc}</pre>'
            '</details>'
        )

    with chat:
        _disp(_HTML(
            '<div style="color:#495057;font-size:13px;font-family:system-ui,sans-serif;padding:2px 0">'
            '<b>Ask me anything about the Zimbabwe DHS 2015 data.</b><br><br>'
            'Try these starters — or go straight to your own question:<br>'
            '<span style="color:#0d6efd">· "Plot LBW rate by province sorted highest to lowest"</span><br>'
            '<span style="color:#0d6efd">· "Compare T3 temperature for LBW vs normal-weight births"</span><br>'
            '<span style="color:#0d6efd">· "Run a logistic regression of LBW on T3 temperature and wealth"</span><br>'
            '<span style="color:#6c757d;font-size:12px">'
            'Each answer remembers the previous — ask follow-ups naturally.</span>'
            '</div>'
        ))

    def _send(_=None):
        q = q_input.value.strip()
        if not q:
            return
        q_input.value = ''
        send_btn.disabled = True

        with chat:
            _disp(_HTML(_user_bubble(q)))
        with status:
            status.clear_output(wait=True)
            _disp(_HTML(
                '<div style="color:#6c757d;font-size:12px;font-family:system-ui,sans-serif;'
                'padding:2px 12px">🤖 <i>thinking…</i></div>'
            ))

        payload = _json.dumps({'question': q, 'history': _hist}).encode()
        try:
            req = _ureq.Request(_url, data=payload,
                                headers={'Content-Type': 'application/json'}, method='POST')
            with _ureq.urlopen(req, timeout=60) as resp:
                res = _json.loads(resp.read())

            _hist.clear()
            _hist.extend(res.get('history', []))
            status.clear_output()

            txt = (res.get('output') or '').strip()
            with chat:
                if txt:
                    _disp(_HTML(_bot_bubble(txt)))
                for fig in res.get('figures', []):
                    _disp(_Img(data=_b64.b64decode(fig)))
                code = (res.get('code') or '').strip()
                if code:
                    _disp(_HTML(_code_block(code)))
                if res.get('error') and not txt and not res.get('figures'):
                    _disp(_HTML(_bot_bubble(f'⚠️ {res["error"]}')))
        except Exception as e:
            status.clear_output()
            with chat:
                _disp(_HTML(_bot_bubble(f'⚠️ Could not reach GeoAI: {e}')))
        finally:
            send_btn.disabled = False

    def _new(_):
        _hist.clear()
        chat.clear_output()
        status.clear_output()
        send_btn.disabled = False  # re-enable in case a previous request was still in flight
        with chat:
            _disp(_HTML(
                '<div style="text-align:center;color:#6c757d;font-size:12px;'
                'font-family:system-ui,sans-serif;padding:6px 0">'
                '── New conversation ──</div>'
            ))

    send_btn.on_click(_send)
    new_btn.on_click(_new)
    q_input.continuous_update = False
    q_input.observe(lambda change: _send() if change.get("new") else None, names="value")

    return _w.VBox(
        [header, chat, status, footer],
        layout=_w.Layout(
            border='1px solid #dee2e6', border_radius='8px',
            width='100%', box_shadow='0 2px 10px rgba(0,0,0,0.07)',
        ),
    )

from IPython.display import clear_output as _co; _co(wait=True)
display(make_geoai_chat())

---
## Synthesis — bringing it all together

You have now run the full GeoAI pipeline: spatial mapping, ecological correlation,
individual-level logistic regression, and AI-assisted exploration.

### Exercise — Write your epidemiological summary *(4 points)*

Write a 4–6 sentence summary addressing:
1. The spatial distribution of LBW across Zimbabwe — which provinces are highest and why
2. Which environmental factors show the clearest association with LBW and at which level
3. Why environmental effects were weaker in the individual-level regression
4. Two key study limitations and how they could be addressed in future work

*(Write your answer in the cell below.)*


*(Write your 4–6 sentence synthesis here.)*


---
## Workshop recap — What we built

| Notebook | What | Tools |
|----------|------|-------|
| NB01 | Loaded and mapped real Zimbabwe DHS 2015 birth records | pandas, matplotlib |
| NB02 | Spatial linkage: joined health records to the environmental grid (cKDTree) | scipy |
| NB03 | Trimester exposure windows; linked pregnancy timing to Zimbabwe's seasonal climate | pandas |
| NB04 | Environmental epidemiology: cluster maps, forest plot, GeoAI assistant | statsmodels, GeoAI |

**The central message:** Satellite-derived environmental data can be linked to population
health records at scale. The associations found here — greener provinces have lower LBW;
hotter clusters have higher LBW — are consistent with biological pathways (heat stress,
nutritional stress via vegetation). But study design and data quality determine what we
can detect. Confounding, selection bias, and exposure measurement error are research
design problems that cannot be solved by better satellite data alone.

---
*Data: Zimbabwe Demographic and Health Survey 2015 (DHS Program). Climate: simulated
exposure grids calibrated to Zimbabwe seasonal patterns and El Niño 2015 conditions.*


In [ ]:
%%html
<div id="nb04-scoreboard" style="font-family:system-ui,sans-serif;border:2px solid #adb5bd;border-radius:10px;padding:20px 24px;margin:24px 0;background:#f8f9fa">
  <div style="font-size:17px;font-weight:700;margin-bottom:12px;color:#1a1a2e">&#128202; Module 4 &#8212; GeoAI Implementation &#8212; Quiz Results</div>
  <div id="nb04-score-display"><p style="color:#856404;background:#fff3cd;padding:10px 12px;border-radius:6px;margin:0">&#9203; Complete all reflection questions above to see your score.</p></div>
  <div id="nb04-next-action" style="margin-top:14px"></div>
</div>
<script>
(function(){
  var QUESTIONS=[{id:'nb04q1',correct:'B'},{id:'nb04q2',correct:'B'},{id:'nb04q3',correct:'B'},{id:'nb04q4',correct:'C'}];
  var PASS=75;
  var NEXT=null;
  var NEXT_LABEL=null;
  var NBID='nb04';
  function update(){
    var correct=0,answered=0;
    QUESTIONS.forEach(function(q){
      var ans=localStorage.getItem('mcq_'+q.id);
      if(ans!==null){answered++;if(ans===q.correct)correct++;}
    });
    var total=QUESTIONS.length;
    var pct=total?Math.round(correct/total*100):0;
    var passed=pct>=PASS;
    var disp=document.getElementById(NBID+'-score-display');
    var act=document.getElementById(NBID+'-next-action');
    if(!disp)return;
    if(answered<total){
      disp.innerHTML='<p style="color:#856404;background:#fff3cd;padding:10px 12px;border-radius:6px;margin:0">&#9203; Answer all '+total+' questions above to see your score. ('+(total-answered)+' remaining)</p>';
      if(act)act.innerHTML='';
    }else if(passed){
      disp.innerHTML='<div style="background:#d4edda;color:#155724;padding:12px 16px;border-radius:8px"><b>&#10003; Score: '+correct+'/'+total+' ('+pct+'%) &#8212; MODULE PASSED</b><br><small>Your progress is saved &#8212; you can close and return to this notebook.</small></div>';
      if(act){
        localStorage.setItem(NBID+'_passed','true');
        localStorage.setItem('nb04_passed','true');
        if(NEXT){act.innerHTML='<a href="'+NEXT+'" style="display:inline-block;background:#198754;color:#fff;padding:10px 22px;border-radius:6px;text-decoration:none;font-weight:600;font-size:14px;margin-top:8px">&#10145; Continue to '+NEXT_LABEL+'</a>';}
        else{act.innerHTML='<div style="background:#cfe2ff;color:#084298;padding:12px 16px;border-radius:8px;margin-top:8px"><b>&#127881; You have completed the full GeoAI pipeline course! Congratulations.</b></div>';}
      }
    }else{
      disp.innerHTML='<div style="background:#f8d7da;color:#721c24;padding:12px 16px;border-radius:8px"><b>&#10007; Score: '+correct+'/'+total+' ('+pct+'%) &#8212; Below pass mark ('+PASS+'%)</b><br><small>Go back to any question and change your answer, then your score updates automatically.</small></div>';
      if(act)act.innerHTML='';
    }
  }
  document.addEventListener('mcq-update',update);
  if(document.readyState==='loading')document.addEventListener('DOMContentLoaded',update);
  else update();
  setInterval(update,3000);
})();
</script>

In [ ]:
import pathlib as _pl
# ── Save progress — run this after passing the quiz above ─────────────────
_badge = _pl.Path.home() / '.nb04_complete'
_badge.touch()
print(f"\u2705  Progress saved: {_badge}")
print("   You have completed the GeoAI for Climate, Environment and Health pipeline!")


---

## You have completed the GeoAI for Climate, Environment and Health pipeline

Starting from a raw DHS birth records file, you have:

1. **Cleaned and mapped** geocoded health data (NB01)
2. **Spatially joined** birth records to climate grid points (NB02)
3. **Constructed** trimester-specific exposure windows (NB03)
4. **Mapped, modelled, and queried** climate–health associations with GeoAI (NB04)

The methods you practised here are transferable to any setting where satellite data
meets health records. Remote sensing professionals now have a route into health research.
Health researchers now have a route into Earth observation.

**That is the bridge this course was built to create.**
